# 01 Daten Laden

### Einführung und Projektüberblick
Dieses Projekt untersucht die Vorhersage der Anzahl von Likes für Instagram-Posts unter Verwendung von maschinellen Lernverfahren. Dabei werden sowohl Metadaten (z.B. Follower-Anzahl, Verifizierungsstatus) als auch visuelle Merkmale aus den Bildern selbst analysiert. Der methodische Ansatz umfasst verschiedene Modelltypen:

Ordinary Least Squares (OLS) Regression
Random Forest Regression
Convolutional Neural Network (CNN) basierte Bildanalyse
Stacking-Modell (Kombination der verschiedenen Ansätze)
Der Workflow folgt einer strukturierten Pipeline von der Datenerfassung bis zur Modellbewertung und -kombination.

#### Voraussetzungen
* **Bibliotheken:** `kaggle`, `os`, `shutil` (`pip install kaggle`).
* **Kaggle API Key:** Eine `kaggle.json` API-Schlüsseldatei im Verzeichnis `~/.kaggle/` ist **zwingend erforderlich**. 
Ohne sie schlägt die Authentifizierung im Download-Schritt fehl. Details zur Einrichtung finden Sie in der [Kaggle Dokumentation](https://www.kaggle.com/docs/api) oder im Projektverzeichnis in der README.md Datei.

In [ ]:
import json
import lzma
import os
import pandas as pd

from kaggle.api.kaggle_api_extended import KaggleApi
from pathlib import Path
from PIL import Image

In [ ]:
# Arbeitsverzeichnis auf das Projektverzeichnis setzen
base_dir = Path.cwd()
if base_dir.name == "notebooks":
    base_dir = base_dir.parent
    os.chdir(base_dir)

### Datengrundlage und Import
Die Daten stammen aus dem Kaggle-Dataset "instagram-posts-dataset" und wurden über die Kaggle API bezogen. Das Dataset umfasst:
- JSON-Dateien mit Metadaten von Instagram-Posts
- Zugehörige Bilddateien im JPG-Format

Der Import-Prozess extrahiert Informationen aus dem "node"-Schlüssel der JSON-Dateien und kombiniert sie mit den Bildinformationen:

In [ ]:
DATASET_ID = "thecoderenroute/instagram-posts-dataset"

data_dir = base_dir / "daten" / "rohdaten" / "instagram-posts-dataset"
os.makedirs(data_dir, exist_ok=True)

data_exists = False
valid_files = [f for f in os.listdir(data_dir) if not f.startswith('.')]
if os.path.exists(data_dir) and valid_files:
    try:
        if len(valid_files) > 0:
             data_exists = True
             print(f"Datensatz bereits vorhanden unter: {data_dir}")
        else:
             print(f"Hinweis: Zielordner '{data_dir}' existiert, ist aber leer.")
    except OSError:
         print(f"Hinweis: Pfad '{data_dir}' existiert, aber Inhalt konnte nicht gelesen werden.")

# ___DOWNLOAD___
if not data_exists:
    print(f"Datensatz nicht oder nur leer vorhanden unter '{data_dir}'. Starte Download via Kaggle API...")
    try:
        print("Initialisiere Kaggle API...")
        api = KaggleApi()
        api.authenticate() 
        print("Authentifizierung erfolgreich.")
        print(f"Lade '{DATASET_ID}' nach '{data_dir}' herunter und entpacke...")
        api.dataset_download_files(DATASET_ID, path=data_dir, unzip=True)
        print("Download und Entpacken abgeschlossen.")

    except ImportError:
         print(f"**FEHLER:** Bibliothek 'kaggle' nicht gefunden. Bitte installieren: pip install kaggle")
    except Exception as e:
        print(f"**FEHLER** beim Download via Kaggle API: {e}")
else:
    print("Datensatz bereits vorhanden.")
for file in os.listdir(data_dir):
    print(f"Gefundene Datei: {file}")

In [ ]:
all_data = []
all_pictures = []
# durch alle Unterordner iterieren
i = 0
for dir in os.listdir(data_dir / "Data"):
    i = i + 1
    dir_dir = data_dir / "Data" / dir
    for file in os.listdir(dir_dir):
        try:
            if file.endswith(".json.xz"):
                file_path = dir_dir / file
                # Entpacken und Lesen der JSON.xz-Datei
                with lzma.open(file_path, mode='rt', encoding='utf-8') as f:
                    data = json.load(f)
                    if "node" in data:
                        data_numbered = {
                            "post_number": i,
                            "folder_name": dir 
                        }
                        data_numbered = data_numbered | data["node"]
                        all_data.append(data_numbered)
            if file.endswith(".jpg"):
                file_path = dir_dir / file
                # Lesen der jpg-Datei
                with Image.open(file_path, mode='r') as f:
                    all_pictures.append({
                        "post_number": i, 
                        "folder_name": dir, 
                        "filename": file, 
                        "format": f.format, 
                        "mode": f.mode, 
                        "size": f.size, 
                        "width": f.width, 
                        "height": f.height, 
                        "file_path": file_path
                    })
        except Exception as e:
            print(f"Fehler beim Verarbeiten der Datei {file_path}: {e}")

# DataFrame aus den gesammelten Daten
df = pd.json_normalize(all_data)
pf = pd.json_normalize(all_pictures)

In [ ]:
# Metadaten und Bildinformationen zusammenführen
merged_df = pd.merge(df, pf, on=['post_number', 'folder_name'])

# Nach Ordnern gruppieren 
by_folder = merged_df.groupby('folder_name')
merged_df.head()

### Datenspeicherung
Die extrahierten Daten wurden als Pickle-Datei (instagram_node_dataframe.pkl) gespeichert, um redundante Verarbeitung zu vermeiden.

In [ ]:
os.makedirs("ergebnisse", exist_ok=True)

df.to_pickle("ergebnisse/instagram_node_dataframe.pkl")
